# Electrolyte generalization benchmark v0.2: example usage

This notebook shows how to (1) inspect the curated data and its provenance, (2) read the frozen evaluation design,
(3) evaluate **your own model** on a frozen task with the benchmark's metrics, and (4) trace an error back to the raw record.

Run `uv run elbench all` first so the curated tables, splits and baseline results exist.

In [1]:
import os, pathlib
import numpy as np
import pandas as pd
from elbench.config import load_config
from elbench.curate import load_processed

root = pathlib.Path.cwd() if (pathlib.Path.cwd() / "configs").exists() else pathlib.Path.cwd().parent
cfg = load_config(root / "configs" / "benchmark_v0.2.yaml")
meas, comps = load_processed(cfg)
pd.set_option("display.width", 160)
meas.inclusion_status.value_counts()

inclusion_status
included    18182
excluded      678
Name: count, dtype: int64

## 1. Canonical records and provenance
One row per reported measurement. Excluded rows stay visible with a reason; every value keeps its original unit and a link to the raw record.

In [2]:
cols = ["measurement_id", "source_doi", "chemistry_system_id", "ratio_basis", "concentration", "concentration_basis",
        "temperature_K", "conductivity_mS_cm", "conductivity_original", "conductivity_unit_original", "track_id", "source_record_link"]
meas[meas.inclusion_status == "included"][cols].sample(5, random_state=0)

,measurement_id,source_doi,chemistry_system_id,ratio_basis,concentration,concentration_basis,temperature_K,conductivity_mS_cm,conductivity_original,conductivity_unit_original,track_id,source_record_link
12614,calisol23:12614,10.1016/0378-7753(91)80004-H,LiAsF6|DME+PC,v,4.000000,molarity,253.150000,0.080000,0.080000,mS/cm,molar_v,https://github.com/Pele0599/CALiSol-23/blob/02...
13930,rahmanian2023:105,10.1038/s41597-023-01936-3,LiPF6|EC+EMC+PC,w,1.542489,molality,333.150000,14.642888,0.014643,S/cm,molal_w,https://zenodo.org/records/7244939/files/Condu...
4916,calisol23:4916,10.1149/1.1513556,LiPF6|EC+EMC+PC+TFP,w,1.320500,molality,323.150000,8.140745,8.140745,mS/cm,molal_w,https://github.com/Pele0599/CALiSol-23/blob/02...
3662,calisol23:3662,10.1149/1.1344281,LiCF3SO3|PC,mol,1.000000,molarity,373.969269,5.469291,5.469291,mS/cm,molar_x,https://github.com/Pele0599/CALiSol-23/blob/02...
14899,rahmanian2023:1074,10.1038/s41597-023-01936-3,LiPF6|EC+EMC+PC,w,1.992977,molality,333.150000,10.955593,0.010956,S/cm,molal_w,https://zenodo.org/records/7244939/files/Condu...


In [3]:
meas[meas.inclusion_status == "excluded"].exclusion_reason.value_counts()

exclusion_reason
missing_conductivity                  523
unresolved_component                   96
negative_conductivity_nonphysical      27
source_quarantine                      23
reported_zero_conductivity              8
negative_concentration_nonphysical      1
Name: count, dtype: int64[pyarrow]

In [4]:
inc = meas[meas.inclusion_status == "included"]
inc.groupby("track_id").agg(measurements=("measurement_id", "size"), formulations=("formulation_id", "nunique"),
                            source_groups=("source_group_id", "nunique"), chemistry_systems=("chemistry_system_id", "nunique"))

,measurements,formulations,source_groups,chemistry_systems
track_id,,,,
molal_w,15516,1752,8,21
molar_v,1269,212,5,88
molar_x,1397,105,9,28


## 2. Frozen evaluation design
Each fold carries an interpretation label. `combined_shift` means chemistry, source or conditions change together, so a failure cannot be attributed to one factor.

In [5]:
folds = pd.read_csv(cfg.p("splits") / "folds.csv")
folds[folds.track_id == "molal_w"][["task_id", "fold_id", "n_train", "n_test", "interpretation_label", "n_formulations", "n_source_groups"]]

,task_id,fold_id,n_train,n_test,interpretation_label,n_formulations,n_source_groups
0,interp_formulation,interp/0,8419,2062,supported_comparison,327,7
1,interp_formulation,interp/1,8599,1882,supported_comparison,317,7
2,interp_formulation,interp/2,8254,2227,supported_comparison,335,7
3,interp_formulation,interp/3,8299,2182,supported_comparison,334,7
4,interp_formulation,interp/4,8362,2119,supported_comparison,334,7
15,source_transfer,source/doi:10.1149/1.1403730,10156,100,supported_comparison,26,1
16,source_transfer,source/doi:10.1149/1.1833611,8041,292,supported_comparison,28,1
17,source_transfer,source/doi:10.1149/1.1914757,9671,100,supported_comparison,20,1
18,source_transfer,source/doi:10.1149/2.0271802jes,10153,60,supported_comparison,11,1
24,author_transfer,authors/AG_ARL_Ding_Jow,373,100,supported_comparison,26,1


## 3. Evaluate your own model
`iter_folds` yields features and training labels for each outer fold, plus nested development folds for tuning.
Test labels are never handed out; `score_predictions` joins them by ID and compares against the stored baselines with a paired bootstrap over groups.

The example model below is a random forest that tunes `min_samples_leaf` on the development folds only.

In [6]:
from sklearn.ensemble import RandomForestRegressor
from elbench.submission import iter_folds, score_predictions

TASK, TRACK = "source_transfer", "molal_w"

def fit(X, y, leaf):
    return RandomForestRegressor(n_estimators=100, min_samples_leaf=leaf, random_state=0, n_jobs=-1).fit(X, np.log10(y))

preds = []
for fold in iter_folds(cfg, TASK, TRACK):
    # choose the leaf size on development folds (other held-out sources inside the training data)
    scores = {}
    for leaf in (1, 5, 20):
        dev_err = [np.mean(np.abs(fit(d.X_train, d.y_train, leaf).predict(d.X_test) - np.log10(d.y_dev))) for d in fold.dev_folds]
        scores[leaf] = np.mean(dev_err)
    best = min(scores, key=scores.get)
    model = fit(fold.X_train, fold.y_train, best)
    preds.append(fold.prediction_frame(10 ** model.predict(fold.X_test)))
    print(f"{fold.fold_id:45s} leaf={best:2d}  n_test={len(fold.X_test)}")
submission = pd.concat(preds)

source/doi:10.1149/1.1403730                  leaf= 1  n_test=100


source/doi:10.1149/1.1833611                  leaf= 1  n_test=292


source/doi:10.1149/1.1914757                  leaf= 1  n_test=100


source/doi:10.1149/2.0271802jes               leaf= 1  n_test=60


In [7]:
result = score_predictions(cfg, TASK, TRACK, submission)
result["summary"][["n_rows", "n_groups", "group_mean_mae_log10", "sample_mae_log10", "worst_quartile_mae_log10", "invalid_rate"]]

,n_rows,n_groups,group_mean_mae_log10,sample_mae_log10,worst_quartile_mae_log10,invalid_rate
0,552,4,0.083392,0.106495,0.14452,0.0


In [8]:
# paired bootstrap over held-out sources; diff_vs_reference is baseline minus submission (positive = submission better)
result["comparison"][["model", "n_groups", "group_mean_mae_log10", "ci_low", "ci_high", "diff_vs_reference", "diff_ci_low", "diff_ci_high"]]

,model,n_groups,group_mean_mae_log10,ci_low,ci_high,diff_vs_reference,diff_ci_low,diff_ci_high
0,extra_trees,4,0.073946,0.034787,0.113105,-0.009446,-0.013337,-0.006308
1,hist_gb,4,0.094987,0.055513,0.130877,0.011594,0.003295,0.023071
2,knn,4,0.075725,0.038428,0.113022,-0.007667,-0.013426,-0.002255
3,median,4,0.430703,0.386588,0.474817,0.347311,0.283578,0.411599
4,ridge,4,0.137085,0.112162,0.162009,0.053693,-0.001065,0.096210
5,ridge_raw,4,0.087958,0.068841,0.107074,0.004565,-0.025570,0.030544
6,submission,4,0.083392,0.042757,0.124027,0.000000,0.000000,0.000000


## 4. Trace an error back to its raw record

In [9]:
g = result["groups"].sort_values("mae_log10", ascending=False)
worst_source = g.group.iat[0]
print("worst held-out source:", worst_source, "MAE log10 =", round(g.mae_log10.iat[0], 3))
sub = submission.merge(meas[["measurement_id", "source_group_id", "conductivity_mS_cm", "temperature_original", "concentration_original",
                             "conductivity_original", "conductivity_unit_original", "source_record_link"]], on="measurement_id")
sub = sub[sub.source_group_id == worst_source]
sub["abs_log_err"] = np.abs(np.log10(sub.prediction_mS_cm) - np.log10(sub.conductivity_mS_cm))
sub.sort_values("abs_log_err", ascending=False).head(5)

worst held-out source: doi:10.1149/1.1833611 MAE log10 = 0.145


,fold_id,measurement_id,prediction_mS_cm,status,source_group_id,conductivity_mS_cm,temperature_original,concentration_original,conductivity_original,conductivity_unit_original,source_record_link,abs_log_err
210,source/doi:10.1149/1.1833611,calisol23:8861,0.265147,ok,doi:10.1149/1.1833611,0.000585,233.75,0.593443,0.000585,mS/cm,https://github.com/Pele0599/CALiSol-23/blob/02...,2.655975
261,source/doi:10.1149/1.1833611,calisol23:8917,0.450138,ok,doi:10.1149/1.1833611,0.002342,243.65,0.983607,0.002342,mS/cm,https://github.com/Pele0599/CALiSol-23/blob/02...,2.283774
251,source/doi:10.1149/1.1833611,calisol23:8906,0.455237,ok,doi:10.1149/1.1833611,0.018735,243.65,0.891803,0.018735,mS/cm,https://github.com/Pele0599/CALiSol-23/blob/02...,1.385576
359,source/doi:10.1149/1.1833611,calisol23:9935,0.285357,ok,doi:10.1149/1.1833611,0.032258,233.75,0.717647,0.032258,mS/cm,https://github.com/Pele0599/CALiSol-23/blob/02...,0.946750
348,source/doi:10.1149/1.1833611,calisol23:9924,0.262650,ok,doi:10.1149/1.1833611,0.032258,233.75,0.623529,0.032258,mS/cm,https://github.com/Pele0599/CALiSol-23/blob/02...,0.910739


The baseline report (`reports/v0.2/report.html`) contains the full results, the source-familiarity investigation and three inspected failure cases.
The coverage audit (`reports/v0.2/coverage.html`) shows which comparisons the data can support.